# 🧠 Text Generation with Vanilla RNN, LSTM & GRU

**Objective:** Design and implement deep learning models capable of learning the underlying structure, grammar, and contextual dependencies of a text corpus to generate coherent and meaningful text sequences.

---

## 📋 Table of Contents
1. [Setup & Imports](#1-setup)
2. [Corpus & Preprocessing](#2-corpus)
3. [Dataset & DataLoader](#3-dataset)
4. [Model Architectures](#4-models)
   - 4.1 Vanilla RNN
   - 4.2 LSTM
   - 4.3 GRU
5. [Training Pipeline](#5-training)
6. [Text Generation](#6-generation)
7. [Comparison & Analysis](#7-analysis)
8. [Visualizations](#8-viz)
9. [Observations & Conclusions](#9-conclusions)

## 1. Setup & Imports <a id='1-setup'></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import pandas as pd
import time
import math
import random
import re
from collections import Counter
from IPython.display import display

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

ModuleNotFoundError: No module named 'torch'

## 2. Corpus & Preprocessing <a id='2-corpus'></a>

We use a rich literary corpus — Shakespeare's works — which has diverse vocabulary, complex grammar, and meaningful long-range dependencies. This makes it an ideal benchmark for comparing RNN variants.

In [ ]:
import urllib.request

# ── Download corpus ──────────────────────────────────────────────────────────
URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
with urllib.request.urlopen(URL) as f:
    raw_text = f.read().decode('utf-8')

print(f'Corpus length : {len(raw_text):,} characters')
print(f'\nSample (first 300 chars):\n{"-"*50}')
print(raw_text[:300])

In [ ]:
# ── Build character-level vocabulary ─────────────────────────────────────────
# We work at the CHARACTER level so the models must learn spelling, grammar,
# punctuation, and style from scratch — a harder and more revealing task.

chars = sorted(set(raw_text))
VOCAB_SIZE = len(chars)

char2idx = {ch: i for i, ch in enumerate(chars)}
idx2char = {i: ch for i, ch in enumerate(chars)}

# Encode the full corpus
encoded = np.array([char2idx[c] for c in raw_text], dtype=np.int64)

print(f'Vocabulary size : {VOCAB_SIZE} unique characters')
print(f'Encoded length  : {len(encoded):,} tokens')
print(f'\nCharacter set   : {"".join(chars)}')

In [ ]:
# ── Exploratory Analysis ──────────────────────────────────────────────────────
freq = Counter(raw_text)
top20 = freq.most_common(20)

labels = [repr(c) for c, _ in top20]
counts = [n for _, n in top20]

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Character frequency
axes[0].bar(labels, counts, color='steelblue', edgecolor='white')
axes[0].set_title('Top-20 Character Frequencies', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Character')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Corpus split
TRAIN_SPLIT = 0.8
VAL_SPLIT   = 0.1
n = len(encoded)
n_train = int(n * TRAIN_SPLIT)
n_val   = int(n * VAL_SPLIT)
n_test  = n - n_train - n_val
axes[1].pie([n_train, n_val, n_test],
            labels=['Train', 'Val', 'Test'],
            autopct='%1.1f%%',
            colors=['#4C72B0', '#DD8452', '#55A868'],
            startangle=140)
axes[1].set_title('Corpus Split', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()
print(f'Train: {n_train:,} | Val: {n_val:,} | Test: {n_test:,} characters')

## 3. Dataset & DataLoader <a id='3-dataset'></a>

In [ ]:
SEQ_LEN    = 100   # input sequence length (context window)
BATCH_SIZE = 128

class TextDataset(Dataset):
    """
    Sliding-window character dataset.
    Input  : sequence of SEQ_LEN characters
    Target : the same sequence shifted by 1 (next-char prediction)
    """
    def __init__(self, data: np.ndarray, seq_len: int):
        self.data    = torch.from_numpy(data).long()
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = self.data[idx          : idx + self.seq_len]
        y = self.data[idx + 1      : idx + self.seq_len + 1]
        return x, y


# ── Split ─────────────────────────────────────────────────────────────────────
train_data = encoded[:n_train]
val_data   = encoded[n_train : n_train + n_val]
test_data  = encoded[n_train + n_val :]

train_ds = TextDataset(train_data, SEQ_LEN)
val_ds   = TextDataset(val_data,   SEQ_LEN)
test_ds  = TextDataset(test_data,  SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, drop_last=True, num_workers=2, pin_memory=True)

print(f'Train samples : {len(train_ds):,}')
print(f'Val   samples : {len(val_ds):,}')
print(f'Test  samples : {len(test_ds):,}')

# Sanity check
xb, yb = next(iter(train_loader))
print(f'\nBatch x shape : {xb.shape}  (batch, seq_len)')
print(f'Batch y shape : {yb.shape}  (batch, seq_len)')

## 4. Model Architectures <a id='4-models'></a>

All three models share the same interface:
- **Embedding layer** → maps token indices to dense vectors
- **Recurrent layers** → Vanilla RNN / LSTM / GRU
- **Dropout** → regularisation between stacked layers
- **Linear head** → projects hidden state → vocabulary logits

### Architecture Comparison

| Feature | Vanilla RNN | LSTM | GRU |
|---|---|---|---|
| Gates | None | Input, Forget, Output | Reset, Update |
| Cell state | No | Yes | No |
| Parameters | Fewest | Most | Intermediate |
| Long-range memory | Poor | Excellent | Good |
| Training speed | Fastest | Slowest | Moderate |

In [ ]:
# ── 4.1  Vanilla RNN ──────────────────────────────────────────────────────────
class VanillaRNN(nn.Module):
    """
    Multi-layer Vanilla RNN language model.

    Recurrence:  h_t = tanh(W_ih · x_t + W_hh · h_{t-1} + b)
    Output    :  logits_t = W_out · h_t

    Limitation: suffers from vanishing gradients over long sequences,
    making it hard to capture long-range dependencies.
    """
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers, dropout):
        super().__init__()
        self.model_type  = 'Vanilla RNN'
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn       = nn.RNN(
            input_size  = embed_dim,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0,
            nonlinearity= 'tanh'
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        embed          = self.dropout(self.embedding(x))   # (B, T, E)
        out, hidden    = self.rnn(embed, hidden)           # (B, T, H)
        out            = self.dropout(out)
        logits         = self.fc(out)                      # (B, T, V)
        return logits, hidden

    def init_hidden(self, batch_size):
        return torch.zeros(self.num_layers, batch_size, self.hidden_size, device=DEVICE)


print('VanillaRNN defined')

In [ ]:
# ── 4.2  LSTM ─────────────────────────────────────────────────────────────────
class LSTMModel(nn.Module):
    """
    Multi-layer LSTM language model.

    Gates:
      Forget gate : f_t = σ(W_f · [h_{t-1}, x_t] + b_f)
      Input  gate : i_t = σ(W_i · [h_{t-1}, x_t] + b_i)
      Cell update : g_t = tanh(W_g · [h_{t-1}, x_t] + b_g)
      Output gate : o_t = σ(W_o · [h_{t-1}, x_t] + b_o)
      Cell state  : c_t = f_t ⊙ c_{t-1} + i_t ⊙ g_t
      Hidden      : h_t = o_t ⊙ tanh(c_t)

    The cell state acts as a long-term memory highway, alleviating
    the vanishing gradient problem.
    """
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers, dropout):
        super().__init__()
        self.model_type  = 'LSTM'
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm      = nn.LSTM(
            input_size  = embed_dim,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        embed          = self.dropout(self.embedding(x))
        out, hidden    = self.lstm(embed, hidden)
        out            = self.dropout(out)
        logits         = self.fc(out)
        return logits, hidden

    def init_hidden(self, batch_size):
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=DEVICE)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=DEVICE)
        return (h0, c0)


print('LSTMModel defined')

In [ ]:
# ── 4.3  GRU ──────────────────────────────────────────────────────────────────
class GRUModel(nn.Module):
    """
    Multi-layer GRU language model.

    Gates:
      Update gate : z_t = σ(W_z · [h_{t-1}, x_t])
      Reset  gate : r_t = σ(W_r · [h_{t-1}, x_t])
      Candidate  : h̃_t = tanh(W · [r_t ⊙ h_{t-1}, x_t])
      Hidden     : h_t  = (1 - z_t) ⊙ h_{t-1} + z_t ⊙ h̃_t

    GRU merges the cell and hidden state into one, using fewer parameters
    than LSTM while retaining most of its expressive power.
    """
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers, dropout):
        super().__init__()
        self.model_type  = 'GRU'
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru       = nn.GRU(
            input_size  = embed_dim,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        embed          = self.dropout(self.embedding(x))
        out, hidden    = self.gru(embed, hidden)
        out            = self.dropout(out)
        logits         = self.fc(out)
        return logits, hidden

    def init_hidden(self, batch_size):
        return torch.zeros(self.num_layers, batch_size, self.hidden_size, device=DEVICE)


print('GRUModel defined')

In [ ]:
# ── Hyperparameters & model instantiation ─────────────────────────────────────
EMBED_DIM   = 128
HIDDEN_SIZE = 512
NUM_LAYERS  = 2
DROPOUT     = 0.3

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

models = {
    'Vanilla RNN': VanillaRNN(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(DEVICE),
    'LSTM'       : LSTMModel (VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(DEVICE),
    'GRU'        : GRUModel  (VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(DEVICE),
}

print(f'{'Model':<15} {'Parameters':>15}')
print('-' * 32)
for name, model in models.items():
    print(f'{name:<15} {count_params(model):>15,}')

## 5. Training Pipeline <a id='5-training'></a>

In [ ]:
def detach_hidden(hidden):
    """Detach hidden state from computation graph (truncated BPTT)."""
    if isinstance(hidden, tuple):          # LSTM: (h, c)
        return tuple(h.detach() for h in hidden)
    return hidden.detach()


def run_epoch(model, loader, criterion, optimizer=None, clip=5.0, is_train=True):
    """
    One pass over the data loader.
    Returns (avg_loss, perplexity).
    """
    model.train() if is_train else model.eval()
    total_loss = 0.0
    total_steps = 0

    with torch.set_grad_enabled(is_train):
        hidden = None
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            # Re-init hidden at every batch to avoid stale graphs
            # (equivalent to non-stateful training)
            hidden = model.init_hidden(x.size(0))

            logits, hidden = model(x, hidden)
            hidden = detach_hidden(hidden)

            # Reshape: (B, T, V) → (B*T, V) and (B, T) → (B*T,)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), y.reshape(-1))

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), clip)  # gradient clipping
                optimizer.step()

            total_loss  += loss.item()
            total_steps += 1

    avg_loss = total_loss / total_steps
    perplexity = math.exp(avg_loss)
    return avg_loss, perplexity


print('Training utilities defined')

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
EPOCHS        = 20
LEARNING_RATE = 3e-3
CLIP          = 5.0
PATIENCE      = 4     # early stopping patience

criterion = nn.CrossEntropyLoss()

# Storage for metrics
history = {name: {'train_loss': [], 'val_loss': [], 'train_ppl': [], 'val_ppl': [],
                  'epoch_time': []}
           for name in models}

# ── Training loop ─────────────────────────────────────────────────────────────
for model_name, model in models.items():
    print(f'\n{"="*60}')
    print(f'  Training  {model_name}')
    print(f'{"="*60}')

    optimizer   = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler   = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2,
                                                        factor=0.5, verbose=True)
    best_val    = float('inf')
    wait        = 0
    best_state  = None

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()

        tr_loss, tr_ppl = run_epoch(model, train_loader, criterion, optimizer,
                                     clip=CLIP, is_train=True)
        vl_loss, vl_ppl = run_epoch(model, val_loader, criterion,
                                     is_train=False)
        elapsed = time.time() - t0

        history[model_name]['train_loss'].append(tr_loss)
        history[model_name]['val_loss'].append(vl_loss)
        history[model_name]['train_ppl'].append(tr_ppl)
        history[model_name]['val_ppl'].append(vl_ppl)
        history[model_name]['epoch_time'].append(elapsed)

        scheduler.step(vl_loss)

        print(f'  Epoch {epoch:02d}/{EPOCHS} | '
              f'Train Loss: {tr_loss:.4f} PPL: {tr_ppl:7.2f} | '
              f'Val Loss: {vl_loss:.4f} PPL: {vl_ppl:7.2f} | '
              f'Time: {elapsed:.1f}s')

        # Early stopping
        if vl_loss < best_val:
            best_val   = vl_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f'  Early stopping triggered at epoch {epoch}.')
                break

    # Restore best weights
    model.load_state_dict(best_state)
    model.to(DEVICE)
    print(f'  Best val loss: {best_val:.4f}  PPL: {math.exp(best_val):.2f}')

print('\n All models trained!')

## 6. Text Generation <a id='6-generation'></a>

We use **temperature sampling**: at each step the raw logits are divided by a temperature `τ`, then a character is sampled from the resulting distribution.

- `τ → 0` : greedy / deterministic (high confidence, low diversity)
- `τ = 1` : unmodified model distribution
- `τ > 1` : more random / creative output

In [ ]:
def generate_text(model, seed_text, length=400, temperature=0.8):
    """
    Auto-regressively generate `length` characters starting from `seed_text`.

    Algorithm:
    1. Encode seed → feed through model to build up the hidden state.
    2. At each step sample next char from softmax(logits / τ).
    3. Feed sampled char back as next input.
    """
    model.eval()
    generated = seed_text

    # ── Prime with seed ──────────────────────────────────────────────────────
    seed_ids = [char2idx.get(c, 0) for c in seed_text]
    inp      = torch.tensor(seed_ids, dtype=torch.long).unsqueeze(0).to(DEVICE)  # (1, L)

    with torch.no_grad():
        hidden = model.init_hidden(1)
        _, hidden = model(inp, hidden)

        # ── Auto-regressive generation ────────────────────────────────────────
        last_char = seed_ids[-1]
        for _ in range(length):
            x      = torch.tensor([[last_char]], dtype=torch.long).to(DEVICE)  # (1,1)
            logits, hidden = model(x, hidden)            # (1, 1, V)
            logits = logits.squeeze() / temperature      # (V,)
            probs  = torch.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1).item()
            generated += idx2char[next_idx]
            last_char  = next_idx

    return generated


print('generate_text() defined')

In [ ]:
SEED_TEXT   = 'ROMEO:'
GEN_LENGTH  = 400
TEMPERATURES = [0.5, 0.8, 1.0]

for model_name, model in models.items():
    print(f'\n{"━"*60}')
    print(f'  {model_name}  — Generated Text')
    print(f'{"━"*60}')
    for temp in TEMPERATURES:
        text = generate_text(model, SEED_TEXT, length=GEN_LENGTH, temperature=temp)
        print(f'\n[Temperature = {temp}]')
        print(text)
        print()

## 7. Test Set Evaluation <a id='7-analysis'></a>

In [ ]:
print(f'\n{"Model":<15} {"Test Loss":>12} {"Test PPL":>12} {"Params":>12} {"Avg Time/Epoch":>16}')
print('─' * 70)

results = {}
for model_name, model in models.items():
    test_loss, test_ppl = run_epoch(model, test_loader, criterion, is_train=False)
    avg_time = np.mean(history[model_name]['epoch_time'])
    n_params = count_params(model)
    results[model_name] = {
        'test_loss': test_loss, 'test_ppl': test_ppl,
        'params': n_params, 'avg_epoch_time': avg_time
    }
    print(f'{model_name:<15} {test_loss:>12.4f} {test_ppl:>12.2f} '
          f'{n_params:>12,} {avg_time:>16.1f}s')

## 8. Visualizations <a id='8-viz'></a>

In [ ]:
# ── Plot 1: Training & Validation Loss curves ─────────────────────────────────
colors = {'Vanilla RNN': '#E74C3C', 'LSTM': '#2ECC71', 'GRU': '#3498DB'}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for model_name in models:
    h     = history[model_name]
    eps   = range(1, len(h['train_loss']) + 1)
    col   = colors[model_name]
    axes[0].plot(eps, h['train_loss'], '-o', label=f'{model_name} (train)',
                 color=col, linewidth=2, markersize=4)
    axes[0].plot(eps, h['val_loss'],   '--s', label=f'{model_name} (val)',
                 color=col, linewidth=2, markersize=4, alpha=0.7)

axes[0].set_title('Training & Validation Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# ── Plot 2: Perplexity curves ─────────────────────────────────────────────────
for model_name in models:
    h   = history[model_name]
    eps = range(1, len(h['val_ppl']) + 1)
    axes[1].plot(eps, h['val_ppl'], '-o', label=model_name,
                 color=colors[model_name], linewidth=2, markersize=4)

axes[1].set_title('Validation Perplexity', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Perplexity (lower = better)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 3: Final metric comparison bar charts ────────────────────────────────
model_names = list(results.keys())
test_ppls   = [results[m]['test_ppl']        for m in model_names]
test_losses = [results[m]['test_loss']       for m in model_names]
params      = [results[m]['params'] / 1e6    for m in model_names]   # millions
times       = [results[m]['avg_epoch_time']  for m in model_names]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
pal = [colors[m] for m in model_names]

def bar_plot(ax, values, title, ylabel, fmt='{:.2f}'):
    bars = ax.bar(model_names, values, color=pal, edgecolor='white', width=0.5)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(values)*0.01,
                fmt.format(v), ha='center', va='bottom', fontweight='bold', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, max(values) * 1.15)
    ax.grid(axis='y', alpha=0.3)

bar_plot(axes[0,0], test_ppls,   'Test Perplexity (↓ better)',  'Perplexity')
bar_plot(axes[0,1], test_losses, 'Test Loss (↓ better)',        'Cross-Entropy')
bar_plot(axes[1,0], params,      'Parameter Count (M)',         'Millions', fmt='{:.2f}M')
bar_plot(axes[1,1], times,       'Avg Epoch Time (s)',          'Seconds')

plt.suptitle('Model Comparison Dashboard', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 4: Temperature sensitivity ──────────────────────────────────────────
temps = [0.3, 0.5, 0.8, 1.0, 1.2, 1.5]
PROBE = 'To be, or not to be'

# Compute entropy of generated distribution at each temperature for each model
def generation_entropy(model, seed, temperature, n_chars=200):
    """Measure the average entropy of the predicted distribution."""
    model.eval()
    seed_ids = [char2idx.get(c, 0) for c in seed]
    inp = torch.tensor(seed_ids, dtype=torch.long).unsqueeze(0).to(DEVICE)
    entropies = []
    with torch.no_grad():
        hidden = model.init_hidden(1)
        _, hidden = model(inp, hidden)
        last = seed_ids[-1]
        for _ in range(n_chars):
            x = torch.tensor([[last]], dtype=torch.long).to(DEVICE)
            logits, hidden = model(x, hidden)
            probs = torch.softmax(logits.squeeze() / temperature, dim=-1)
            ent = -(probs * (probs + 1e-10).log()).sum().item()
            entropies.append(ent)
            last = torch.multinomial(probs, 1).item()
    return np.mean(entropies)

fig, ax = plt.subplots(figsize=(10, 5))
for model_name, model in models.items():
    ents = [generation_entropy(model, PROBE, t) for t in temps]
    ax.plot(temps, ents, '-o', label=model_name,
            color=colors[model_name], linewidth=2, markersize=6)

ax.axvline(1.0, color='gray', linestyle='--', alpha=0.5, label='τ = 1 (unscaled)')
ax.set_title('Generation Entropy vs Temperature', fontsize=13, fontweight='bold')
ax.set_xlabel('Temperature (τ)')
ax.set_ylabel('Average Entropy (nats)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('temperature_entropy.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 5: Radar (spider) chart — multi-criteria comparison ──────────────────
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

# Normalise each metric 0→1  (higher = better for radar)
def norm(vals, lower_is_better=True):
    arr = np.array(vals, dtype=float)
    mn, mx = arr.min(), arr.max()
    if mx == mn:
        return np.ones_like(arr)
    n = (arr - mn) / (mx - mn)
    return (1 - n) if lower_is_better else n

categories   = ['Test PPL\n(↓)', 'Test Loss\n(↓)', 'Speed\n(↓time)', 'Compactness\n(↓params)']
N            = len(categories)
angles       = [n / float(N) * 2 * math.pi for n in range(N)]
angles      += angles[:1]

ppls  = norm(test_ppls)
losss = norm(test_losses)
spd   = norm(times)
cmpct = norm(params)

fig, ax = plt.subplots(1, 1, figsize=(7, 7), subplot_kw=dict(polar=True))
for i, model_name in enumerate(model_names):
    vals   = [ppls[i], losss[i], spd[i], cmpct[i]]
    vals  += vals[:1]
    ax.plot(angles, vals, '-o', label=model_name, color=list(colors.values())[i], linewidth=2)
    ax.fill(angles, vals, alpha=0.1, color=list(colors.values())[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10)
ax.set_ylim(0, 1)
ax.set_title('Multi-Criteria Radar Chart\n(closer to edge = better)', fontsize=12,
             fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig('radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Observations & Conclusions <a id='9-conclusions'></a>

In [ ]:
# ── Final summary table ───────────────────────────────────────────────────────
summary_df = pd.DataFrame({
    'Model'        : model_names,
    'Test Loss'    : [f"{results[m]['test_loss']:.4f}"      for m in model_names],
    'Test PPL'     : [f"{results[m]['test_ppl']:.2f}"       for m in model_names],
    'Parameters'   : [f"{results[m]['params']:,}"           for m in model_names],
    'Time/Epoch(s)': [f"{results[m]['avg_epoch_time']:.1f}" for m in model_names],
})

display(summary_df.style.set_caption('Final Model Summary')
                        .set_table_styles([{'selector': 'th',
                                            'props': [('background-color', '#4C72B0'),
                                                      ('color', 'white'),
                                                      ('font-weight', 'bold')]}]))